In [1]:
import os
os.chdir('D:\\ai-engineering-buildcamp-codespace\\Homework\\')
print(os.getcwd())

D:\ai-engineering-buildcamp-codespace\Homework


In [31]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import urlparse
from markitdown import MarkItDown
from typing import Any, Dict, Iterable, List
import json
from pydantic import BaseModel, Field
from typing import Literal

In [3]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

True

In [4]:
from openai import OpenAI
openai_client  = OpenAI()

In [5]:
books = pd.read_csv('books.csv')

In [7]:
books

,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...
5,Think OS,https://greenteapress.com/wp/think-os/,http://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,https://greenteapress.com/wp/think-c/,https://raw.githubusercontent.com/tscheffl/Thi...


In [9]:
#Question: Create a script to download the PDF files and save in the local directory 
# Get total count for progress
total = len(books)

# Download each PDF
for index, row in books.iterrows():
    url = row['pdf_url']
    
    # Extract filename from URL
    filename = Path(urlparse(url).path).name
    
    print(f"[{index + 1}/{total}] Downloading: {filename}")
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Save the PDF
        with open(filename, 'wb') as f:
            f.write(response.content)
        
        print(f"  ✓ Successfully downloaded {filename}")
    
    except Exception as e:
        print(f"  ✗ Failed to download {filename}")
        print(f"  Error: {e}")

print("\nDownload complete! All PDFs saved to current directory.")

[1/7] Downloading: thinkpython2.pdf
  ✓ Successfully downloaded thinkpython2.pdf
[2/7] Downloading: thinkdsp.pdf
  ✓ Successfully downloaded thinkdsp.pdf
[3/7] Downloading: thinkcomplexity2.pdf
  ✓ Successfully downloaded thinkcomplexity2.pdf
[4/7] Downloading: thinkjava2.pdf
  ✓ Successfully downloaded thinkjava2.pdf
[5/7] Downloading: PhysicalModelingInMatlab4.pdf
  ✓ Successfully downloaded PhysicalModelingInMatlab4.pdf
[6/7] Downloading: thinkos.pdf
  ✓ Successfully downloaded thinkos.pdf
[7/7] Downloading: Think-C.pdf
  ✓ Successfully downloaded Think-C.pdf

Download complete! All PDFs saved to current directory.


In [11]:
#Question 2. Converting PDFs to Markdown

# I installed the 'markitdown[pdf]' package to convert PDF files to Markdown format.

# Create output directory
output_dir = Path("books_text")
output_dir.mkdir(exist_ok=True)

# Initialize MarkItDown
md = MarkItDown()

# Get all PDF files in current directory
pdf_files = list(Path(".").glob("*.pdf"))

if not pdf_files:
    print("No PDF files found in current directory!")
else:
    total = len(pdf_files)
    print(f"Found {total} PDF files to convert\n")
    
    # Convert each PDF
    for index, pdf_file in enumerate(pdf_files, 1):
        print(f"[{index}/{total}] Converting: {pdf_file.name}")
        
        try:
            # Convert PDF to markdown
            result = md.convert(str(pdf_file))
            
            # Create output filename (replace .pdf with .md)
            output_file = output_dir / f"{pdf_file.stem}.md"
            
            # Save markdown content
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(result.text_content)
            
            print(f"  ✓ Saved to: {output_file}")
        
        except Exception as e:
            print(f"  ✗ Failed to convert {pdf_file.name}")
            print(f"  Error: {e}")
    
    print(f"\nConversion complete! Markdown files saved to '{output_dir}/' directory.")

Found 7 PDF files to convert

[1/7] Converting: PhysicalModelingInMatlab4.pdf
  ✓ Saved to: books_text\PhysicalModelingInMatlab4.md
[2/7] Converting: Think-C.pdf
  ✓ Saved to: books_text\Think-C.md
[3/7] Converting: thinkcomplexity2.pdf
  ✓ Saved to: books_text\thinkcomplexity2.md
[4/7] Converting: thinkdsp.pdf
  ✓ Saved to: books_text\thinkdsp.md
[5/7] Converting: thinkjava2.pdf
  ✓ Saved to: books_text\thinkjava2.md
[6/7] Converting: thinkos.pdf
  ✓ Saved to: books_text\thinkos.md
[7/7] Converting: thinkpython2.pdf
  ✓ Saved to: books_text\thinkpython2.md

Conversion complete! Markdown files saved to 'books_text/' directory.


In [5]:
def sliding_window(
    seq: Iterable[Any],
    size: int,
    step: int,
) -> List[Dict[str, Any]]:
    """Create overlapping chunks from a sequence using a sliding window approach.

    Args:
        seq: The input sequence (string or list) to be chunked.
        size: The size of each chunk/window.
        step: The step size between consecutive windows.

    Returns:
        A list of dictionaries, each containing:
            - 'start': The starting position of the chunk in the original sequence
            - 'content': The chunk content

    Raises:
        ValueError: If size or step are not positive integers.

    Example:
        >>> sliding_window("hello world", size=5, step=3)
        [{'start': 0, 'content': 'hello'}, {'start': 3, 'content': 'lo wo'}]
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        batch = seq[i : i + size]
        result.append({"start": i, "content": batch})
        if i + size > n:
            break

    return result


def chunk_documents(
    documents: Iterable[Dict[str, str]],
    size: int = 2000,
    step: int = 1000,
    content_field_name: str = "content",
) -> List[Dict[str, str]]:
    """Split a collection of documents into smaller chunks using sliding windows.

    Takes documents and breaks their content into overlapping chunks while preserving
    all other document metadata (filename, etc.) in each chunk.

    Args:
        documents: An iterable of document dictionaries. Each document must have a content field.
        size: The maximum size of each chunk. Defaults to 2000.
        step: The step size between chunks. Defaults to 1000.
        content_field_name: The name of the field containing document content.
            Defaults to 'content'.

    Returns:
        A list of chunk dictionaries. Each chunk contains:
            - All original document fields except the content field
            - 'start': Starting position of the chunk in original content
            - 'content': The chunk content

    Example:
        >>> documents = [{'content': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, size=100, step=50)
        >>> # Or with custom content field:
        >>> documents = [{'text': 'long text...', 'filename': 'doc.txt'}]
        >>> chunks = chunk_documents(documents, content_field_name='text')
    """
    results = []

    for doc in documents:
        doc_copy = doc.copy()
        doc_content = doc_copy.pop(content_field_name)
        chunks = sliding_window(doc_content, size=size, step=step)
        for chunk in chunks:
            chunk.update(doc_copy)
        results.extend(chunks)

    return results

In [6]:
##3. Question chunking for RAG
## here we chunk the documents as lines then convert to strings for indexing in the next stage
## Advantage of chunking by lines : Chunking by lines is more semantically meaningful, Lines often represent logical units (sentences, paragraphs)
#size=100 lines ensures you don't cut mid-sentence, step=50 lines creates overlap for better context retention in RAG retrieval


books_dir = Path('books_text')
md_files = list(books_dir.glob('*.md'))

all_chunks = []

for md_file in md_files:
    print(f"Reading: {md_file.name}")
    
    # Read and clean
    content = md_file.read_text(encoding='utf-8')
    lines = content.splitlines()
    clean_lines = [line for line in lines if line.strip()]
    
    # Create document
    doc = {'content': clean_lines, 'source': md_file.name}
    
    # Chunk this document
    doc_chunks = chunk_documents([doc], size=100, step=50)
    
    # Add chunk_id and append to results
    for chunk_id, chunk in enumerate(doc_chunks):
        chunk['chunk_id'] = chunk_id
        all_chunks.append(chunk)
    
    print(f"  Created {len(doc_chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

Reading: PhysicalModelingInMatlab4.md
  Created 119 chunks
Reading: Think-C.md
  Created 107 chunks
Reading: thinkcomplexity2.md
  Created 144 chunks
Reading: thinkdsp.md
  Created 110 chunks
Reading: thinkjava2.md
  Created 247 chunks
Reading: thinkos.md
  Created 68 chunks
Reading: thinkpython2.md
  Created 214 chunks

Total chunks: 1009


In [14]:
all_chunks[10]

{'start': 500,
 'content': ['ans = 28.2743',
  '| or as an | argument to | a function: |     |     |     |',
  '| -------- | ----------- | ----------- | --- | --- | --- |',
  '>> sin(pi/2)',
  'ans = 1',
  'Whenever you evaluate an expression, MATLAB assigns the result to a variable named ans.',
  'You can use ans in a subsequent calculation as shorthand for “the value of the previous ex-',
  'pression.”',
  '| >> 3^2 | + 4^2 |     |     |     |     |',
  '| ------ | ----- | --- | --- | --- | --- |',
  'ans = 25',
  '>> sqrt(ans)',
  'ans = 5',
  'But keep in mind that the value of ans changes every time you evaluate an expression.',
  '1.3 Variables 7',
  '| 1.3.1 | Assignment |     | Statements |     |     |     |',
  '| ----- | ---------- | --- | ---------- | --- | --- | --- |',
  'You can create your own variables, and give them values, with an assignment statement. The',
  '| assignment | operator |     | is the equals | sign | (=), used | like so: |',
  '| ---------- | -------- |

In [8]:
from minsearch import Index

In [9]:
# Prepare documents for minsearch - convert list of lines to string
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        doc = chunk.copy()
        # Convert list of lines to string
        doc['content'] = "\n".join(chunk['content'])
        documents.append(doc)
    return documents

# Prepare and index
documents = prepare_documents(all_chunks)

In [10]:
documents[0]

{'start': 0,
 'content': '| Physical | Modeling | in MATLAB |\n| -------- | -------- | --------- |\nVersion 4.0\nAllen B. Downey\nGreen Tea Press\nNeedham, Massachusetts\nPhysical Modeling in MATLAB\nCopyright 2012, 2019, 2021 Allen B. Downey\nGreen Tea Press\n9 Washburn Ave\nNeedham MA 02492\nPermissionisgrantedtocopy, distribute, and/ormodifythisdocumentunderthetermsofthe\nCreative Commons Attribution-NonCommercial 4.0 Unported License, which is available at\nhttps://greenteapress.com/matlab/license.\nThis book was typeset by the author using pdflatex, among other free, open-source programs.\nThe LaTeX source for this book is available from https://greenteapress.com/matlab.\nTheinformationinthisbookisdistributedonan“AsIs” basis, withoutwarranty. Whileevery\nprecaution has been taken in the preparation of this work, neither the author nor No Starch\nPress, Inc. shall have any liability to any person or entity with respect to any loss or damage\ncaused or alleged to be caused directly 

In [11]:
# Create and fit the index
index = Index(
    text_fields=["content"],  # Fields to search in
    keyword_fields=["source"]  # Fields for filtering
)

index.fit(documents)

In [12]:
print(f"Number of indexed documents: {len(documents)}")

Number of indexed documents: 1009


In [13]:
print(f"Number of indexed documents: {len(index.docs)}")

Number of indexed documents: 1009


In [21]:
results = index.search("python function definition", num_results=5)

In [22]:
results

[{'start': 1900,
  'content': 'when you are comfortable with Python, I’ll make suggestions for installing Python on your\ncomputer.\nThere are a number of web pages you can use to run Python. If you already have a fa-\nvorite, go ahead and use it. Otherwise I recommend PythonAnywhere. I provide detailed\ninstructions for getting started at http://tinyurl.com/thinkpython2e.\nThere are two versions of Python, called Python 2 and Python 3. They are very similar, so\nif you learn one, it is easy to switch to the other. In fact, there are only a few differences you\nwill encounter as a beginner. This book is written for Python 3, but I include some notes\nabout Python 2.\nThe Python interpreter is a program that reads and executes Python code. Depending\non your environment, you might start the interpreter by clicking on an icon, or by typing\npython on a command line. When it starts, you should see output like this:\nPython 3.4.0 (default, Jun 19 2015, 14:20:21)\n[GCC 4.8.2] on linux\nType

Question 5. Full RAG

In [27]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    #converts your Python list/dict into a formatted JSON string. the seach results are in a list of dictionaries format, 
    #  json.dumps will convert that into a readable string format with indentation for better readability in the prompt.
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
         temperature=0,  # For reproducibility
    )
    # Extract token usage
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    return response.output_text, input_tokens, output_tokens


## this is the main function that ties everything together - it takes a user query, retrieves relevant chunks from the index, builds a prompt, 
# and gets an answer from the LLM
# first the search function is called to get relevant chunks, 
# second the prompt is built using the question and search results,
# third  the llm function is called to get the answer based on the prompt and instructions
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm(prompt, instructions)
    return {
        'answer': answer,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': input_tokens + output_tokens
    }


In [28]:
question = "python function definition"
result = rag(question)

In [29]:
print(f"Answer: {result['answer']}\n")

Answer: In Python, a function is defined using the `def` keyword, followed by the function name and parentheses containing any parameters. The body of the function is indented. Here’s a basic structure of a function definition:

```python
def function_name(parameters):
    # Function body
    # Perform operations
    return result  # Optional return statement
```

### Example:
Here’s a simple example of a function that adds two numbers:

```python
def add_numbers(a, b):
    return a + b
```

In this example:
- `add_numbers` is the name of the function.
- `a` and `b` are parameters.
- The function returns the sum of `a` and `b`.

You can call this function by passing two arguments:

```python
result = add_numbers(3, 5)
print(result)  # Output: 8
```

This is a basic overview of how to define and use functions in Python. If you have more specific questions about functions, feel free to ask!



In [30]:
print(f"Input tokens: {result['input_tokens']}")
print(f"Output tokens: {result['output_tokens']}")
print(f"Total tokens: {result['total_tokens']}")

Input tokens: 7347
Output tokens: 213
Total tokens: 7560


Question 6. Structured vs Unstructured Output

In [32]:
class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [33]:
def llm_structured(user_prompt, output_type, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
         temperature=0,# For reproducibility
          text_format=output_type  
    )
    # Extract token usage
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    return response.output_parsed, input_tokens, output_tokens


def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, input_tokens, output_tokens = llm_structured(user_prompt=prompt, output_type=RAGResponse, instructions=instructions)    
    return  {
        'answer': answer,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': input_tokens + output_tokens
    }

In [34]:
question = "python function definition"
result_structured = rag_structured(question )

In [35]:
print(f"Answer: {result_structured['answer']}\n")

Answer: answer='In Python, a function is defined using the `def` keyword followed by the function name and parentheses. Inside the parentheses, you can specify parameters that the function can accept. The body of the function contains the code that will be executed when the function is called. Here’s a simple example:\n\n```python\ndef my_function(parameter1, parameter2):\n    # This is the body of the function\n    result = parameter1 + parameter2\n    return result\n```\n\nIn this example:\n- `my_function` is the name of the function.\n- `parameter1` and `parameter2` are parameters that the function takes.\n- The function adds the two parameters and returns the result.\n\nTo call the function, you would use:\n```python\noutput = my_function(5, 3)\nprint(output)  # This will print 8\n```\n\nFunctions help organize code, avoid repetition, and manage variable scope effectively.' found_answer=True confidence=0.9 confidence_explanation='The context provided detailed information about func

In [36]:
print(f"Input tokens: {result_structured['input_tokens']}")
print(f"Output tokens: {result_structured['output_tokens']}")
print(f"Total tokens: {result_structured['total_tokens']}")

Input tokens: 7533
Output tokens: 292
Total tokens: 7825
